# Students Performance — Senior Data Analyst Notebook

> **Data integrity rule:** The original dataset is loaded read-only and is never modified.  
> All analysis is performed on derived objects/copies. No records are deleted.

This notebook follows the requested analysis framework. Because the uploaded file is **StudentsPerformance.csv**, not a sales workbook, sales-specific fields such as Sales, Profit, Product, Customer, Region, Price, Cost, Discount, and Date are not assumed or invented. Where a requested sales analysis is not supported by the available columns, the notebook explicitly documents that limitation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

FILE = "/kaggle/input/datasets/spscientist/students-performance-in-exams/StudentsPerformance.csv"
df = pd.read_csv(FILE)

# Preserve the original data unchanged.
original_df = df.copy(deep=True)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
display(df.head())
display(df.info())

In [ ]:
df

## 1. Data Quality Report

In [ ]:
# Structure and data types
quality = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean() * 100).values,
    "unique_values": df.nunique(dropna=True).values
})

# Exact duplicate rows
duplicate_count = df.duplicated().sum()

# Invalid date check: no date-like columns are present.
date_like_cols = [c for c in df.columns if any(k in c.lower() for k in ["date", "time", "day", "month", "year"])]
invalid_dates = {}
for c in date_like_cols:
    parsed = pd.to_datetime(df[c], errors="coerce")
    invalid_dates[c] = int(parsed.isna().sum())

print("Duplicate rows:", duplicate_count)
print("Date-like columns:", date_like_cols if date_like_cols else "None")
print("Invalid dates:", invalid_dates if invalid_dates else "Not applicable")
display(quality)


In [ ]:
# Consistency checks for categorical columns
categorical_cols = df.select_dtypes(include="object").columns.tolist()

for col in categorical_cols:
    values = sorted(df[col].dropna().astype(str).unique())
    print(f"\n{col}: {len(values)} unique values")
    print(values)

# Numeric outlier screening using IQR; outliers are reported, not removed.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
outlier_rows = []

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (df[col] < lower) | (df[col] > upper)
    outlier_rows.append({
        "column": col,
        "Q1": q1,
        "Q3": q3,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(mask.sum()),
        "outlier_pct": float(mask.mean() * 100)
    })

outliers = pd.DataFrame(outlier_rows)
display(outliers)


### Data Quality Conclusions

- Missing values are measured directly from the dataset.
- Exact duplicate rows are checked without deleting them.
- No date-based quality checks are fabricated when no date column exists.
- Categorical values are inspected for inconsistent spelling/casing.
- Numeric outliers are flagged using the IQR rule but **not removed**.
- Any cleaning recommendation below is a recommendation only; the original dataset remains unchanged.


## 2. Sales Analysis / Available-Data EDA

In [ ]:
# Available KPIs from the uploaded dataset
score_cols = ["math score", "reading score", "writing score"]

kpis = pd.Series({
    "Students": len(df),
    "Average Math Score": df["math score"].mean(),
    "Average Reading Score": df["reading score"].mean(),
    "Average Writing Score": df["writing score"].mean(),
    "Average Overall Score": df[score_cols].mean(axis=1).mean(),
    "Math Median": df["math score"].median(),
    "Reading Median": df["reading score"].median(),
    "Writing Median": df["writing score"].median()
})

display(kpis.to_frame("Value"))


In [ ]:
# Group-level performance
group_cols = [
    "gender",
    "race/ethnicity",
    "parental level of education",
    "lunch",
    "test preparation course"
]

for col in group_cols:
    summary = (
        df.groupby(col)[score_cols]
          .mean()
          .assign(Overall=lambda x: x.mean(axis=1))
          .sort_values("Overall", ascending=False)
    )
    print(f"\n### {col}")
    display(summary)

# Strongest and weakest individual dimensions
overall_by_student = df[score_cols].mean(axis=1)
print("Highest overall score:", overall_by_student.max())
print("Lowest overall score:", overall_by_student.min())


In [ ]:
# Evidence-based driver analysis
numeric_corr = df[score_cols].corr()

print("Score correlations:")
display(numeric_corr)

prep = df.groupby("test preparation course")[score_cols].mean()
prep["Overall"] = prep.mean(axis=1)

lunch = df.groupby("lunch")[score_cols].mean()
lunch["Overall"] = lunch.mean(axis=1)

print("Test preparation comparison:")
display(prep)

print("Lunch comparison:")
display(lunch)


### Sales-specific limitation

The uploaded dataset contains student performance data, not transactional sales data. Therefore, the following cannot be calculated without inventing fields:

- Sales revenue
- Profit
- Products
- Customers
- Regions
- Prices
- Costs
- Discounts
- Sales trends

The notebook instead reports the equivalent **available-data performance drivers** using only the actual columns in the dataset.


### Root Cause Analysis Policy

No causal explanation is assumed. A major change can only be attributed to a factor when the factor is actually represented in the data and the comparison supports the claim.

For this dataset there is no time dimension, so a genuine MoM/QoQ/YoY root-cause analysis is **not applicable**.


## 4. Business Analysis

In [ ]:
business_questions = []

# Q1
overall = df[score_cols].mean(axis=1)
business_questions.append({
    "Business Question": "What is the overall performance level?",
    "Answer": f"The mean overall score is {overall.mean():.2f} across {len(df):,} students.",
    "Evidence": "Mean of math, reading, and writing scores."
})

# Q2
prep_overall = df.groupby("test preparation course")[score_cols].mean().mean(axis=1)
best_prep = prep_overall.idxmax()
worst_prep = prep_overall.idxmin()
business_questions.append({
    "Business Question": "Which test-preparation group performs better?",
    "Answer": f"'{best_prep}' has the higher average overall score ({prep_overall[best_prep]:.2f}) versus '{worst_prep}' ({prep_overall[worst_prep]:.2f}).",
    "Evidence": "Grouped means by test preparation course."
})

# Q3
lunch_overall = df.groupby("lunch")[score_cols].mean().mean(axis=1)
business_questions.append({
    "Business Question": "Do the lunch groups differ in performance?",
    "Answer": "Yes, the group averages differ.",
    "Evidence": lunch_overall.to_dict()
})

# Q4
corr_rw = df["reading score"].corr(df["writing score"])
business_questions.append({
    "Business Question": "Which score dimensions move together most strongly?",
    "Answer": f"Reading and writing have correlation {corr_rw:.3f}.",
    "Evidence": "Pearson correlation across student-level scores."
})

# Q5
business_questions.append({
    "Business Question": "Can sales profitability be evaluated?",
    "Answer": "No. Revenue, cost, profit, price, and discount fields are not present.",
    "Evidence": "Dataset schema inspection."
})

display(pd.DataFrame(business_questions))


### Management Insights, Risks, and Opportunities

**Insights**
1. Performance differs across the available demographic and preparation dimensions.
2. Test preparation status is a measurable performance differentiator in this dataset.
3. Reading and writing scores show a very strong positive relationship.

**Risks**
1. The data does not contain time, financial, customer, or transactional fields, limiting business forecasting and profitability analysis.
2. Group differences should not automatically be interpreted as causal relationships.

**Opportunities**
1. Investigate the characteristics of higher-performing groups.
2. Use the three score dimensions together rather than relying on one score alone.
3. Add time and intervention fields if longitudinal performance analysis is required.

**Priority actions**
1. Preserve the raw dataset.
2. Validate categorical definitions with the data owner.
3. Add a time/intervention dimension for trend analysis.
4. If the intended task is sales analysis, provide the actual sales workbook.


## 7. Executive Notebook

In [ ]:
# Presentation-ready executive summary generated from the actual data
overall_mean = overall.mean()
math_mean = df["math score"].mean()
reading_mean = df["reading score"].mean()
writing_mean = df["writing score"].mean()

prep_diff = prep_overall.max() - prep_overall.min()
lunch_diff = lunch_overall.max() - lunch_overall.min()

print("EXECUTIVE HIGHLIGHTS")
print("=" * 80)
print(f"Students analyzed: {len(df):,}")
print(f"Overall average score: {overall_mean:.2f}")
print(f"Math average: {math_mean:.2f}")
print(f"Reading average: {reading_mean:.2f}")
print(f"Writing average: {writing_mean:.2f}")
print(f"Preparation-group overall gap: {prep_diff:.2f} points")
print(f"Lunch-group overall gap: {lunch_diff:.2f} points")
print(f"Reading-Writing correlation: {corr_rw:.3f}")

print("\nMAJOR INSIGHTS")
print(f"- Highest average score dimension: {pd.Series({'Math': math_mean, 'Reading': reading_mean, 'Writing': writing_mean}).idxmax()}")
print(f"- Strongest score relationship: Reading vs Writing (r={corr_rw:.3f})")
print(f"- Higher preparation-group average: {best_prep}")

print("\nROOT CAUSES")
print("- No causal root cause is asserted.")
print("- Group differences are reported only as observed differences.")
print("- Time-based root-cause analysis is unavailable because there is no date field.")

print("\nBUSINESS PROBLEMS")
print("- The file does not contain sales, revenue, cost, profit, customer, product, region, discount, or date fields.")

print("\nOPPORTUNITIES")
print("- Analyze intervention/preparation effects with longitudinal data.")
print("- Segment performance by the available categorical dimensions.")
print("- Add time and business transaction fields if the intended analysis is sales.")

print("\nNEXT STEPS")
print("1. Keep the original dataset unchanged.")
print("2. Validate category definitions with the data owner.")
print("3. Add date/time fields for trend analysis.")
print("4. Upload the intended sales workbook for Sales/Profit analysis.")


In [ ]:
# Final integrity check: prove that the original loaded data was not modified.
assert df.equals(original_df), "Original dataframe was modified."
print("Integrity check passed: original data remains unchanged.")


# Charts

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("/kaggle/input/datasets/spscientist/students-performance-in-exams/StudentsPerformance.csv")

subjects = ["math score", "reading score", "writing score"]
avg_scores = df[subjects].mean().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
avg_scores.plot(kind="bar")

plt.title("Average Score by Subject")
plt.xlabel("Subject")
plt.ylabel("Average Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df["overall_score"] = df[
    ["math score", "reading score", "writing score"]
].mean(axis=1)

prep_analysis = (
    df.groupby("test preparation course")["overall_score"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(8, 5))
prep_analysis.plot(kind="bar")

plt.title("Average Overall Score by Test Preparation")
plt.xlabel("Test Preparation Course")
plt.ylabel("Average Overall Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
lunch_analysis = (
    df.groupby("lunch")["overall_score"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(8, 5))
lunch_analysis.plot(kind="bar")

plt.title("Average Overall Score by Lunch Type")
plt.xlabel("Lunch Type")
plt.ylabel("Average Overall Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(
    df["overall_score"],
    bins=15,
    edgecolor="black"
)

plt.title("Distribution of Overall Student Scores")
plt.xlabel("Overall Score")
plt.ylabel("Number of Students")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    df["reading score"],
    df["writing score"],
    alpha=0.6
)

plt.title("Reading Score vs Writing Score")
plt.xlabel("Reading Score")
plt.ylabel("Writing Score")

plt.tight_layout()
plt.show()

In [ ]:
correlation = df["reading score"].corr(df["writing score"])

print(f"Correlation between Reading and Writing: {correlation:.3f}")

In [ ]:
parent_education = (
    df.groupby("parental level of education")["overall_score"]
    .mean()
    .sort_values(ascending=True)
)

plt.figure(figsize=(10, 6))

parent_education.plot(kind="barh")

plt.title("Average Overall Score by Parental Education")
plt.xlabel("Average Overall Score")
plt.ylabel("Parental Level of Education")

plt.tight_layout()
plt.show()

In [ ]:
race_analysis = (
    df.groupby("race/ethnicity")["overall_score"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(8, 5))

race_analysis.plot(kind="bar")

plt.title("Average Overall Score by Race/Ethnicity")
plt.xlabel("Race/Ethnicity")
plt.ylabel("Average Overall Score")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

score_corr = df[
    ["math score", "reading score", "writing score"]
].corr()

plt.figure(figsize=(7, 5))

sns.heatmap(
    score_corr,
    annot=True,
    fmt=".2f",
    cmap="Blues"
)

plt.title("Correlation Matrix of Student Scores")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

df[
    ["math score", "reading score", "writing score"]
].boxplot()

plt.title("Score Distribution and Outliers")
plt.xlabel("Subject")
plt.ylabel("Score")

plt.tight_layout()
plt.show()

In [ ]:
gender_scores = df.groupby("gender")[
    ["math score", "reading score", "writing score"]
].mean()

gender_scores.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Average Scores by Gender")
plt.xlabel("Gender")
plt.ylabel("Average Score")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()